<!-- NOTEBOOK_METADATA source: "Jupyter Notebook" title: "Prioritize Traces for Human Annotation under a Fixed Review Budget" sidebarTitle: "Annotation Queue Prioritization" description: "Decide which traces go into a Langfuse annotation queue when reviewers can only look at a fixed number per day. Compare random, max-score, severity-weighted, uncertainty and mixed selection on the same candidate pool, push the selection through the API, and close the loop with the human scores." category: "Examples" -->

# Prioritize Traces for Human Annotation under a Fixed Review Budget

[Annotation queues](https://langfuse.com/docs/evaluation/evaluation-methods/annotation-queues) give domain experts a structured way to score traces, observations and sessions. The queue itself is easy to fill: the UI and the [Public API](https://api.reference.langfuse.com/#tag/annotationqueues) both add items. The hard part is upstream. A reviewer team can look at a fixed number of items per day, and every item they look at is one they did not look at instead.

This cookbook treats the human review budget as the constraint and asks: **given the automated signals already attached to your traces, which ones should the budget reach?** It compares five selection policies on the same candidate pool, pushes one of them into an annotation queue through the API, and reads the human scores back to check the pushed selection and the judges against them.

Two of the policies mirror arms of [review-router](https://github.com/XinyangWuEthz/review-router), a reproducible benchmark for human-review routing: `max_score` is its probability ordering and `severity` its default ordering. `random`, `uncertainty` and `mixed` are additions for annotation queues and were not benchmarked. In that benchmark, four orderings (first-in-first-out, probability, severity, band-first) of the same review queue were replayed against the same simulated reviewer capacity. Ordering by *severity* (the highest of probability times a declared weight over all labels) rather than first-in-first-out completed more high-risk reviews per shift and exactly as many fewer other reviews: the queue reallocated a fixed capacity, it did not add any. The same is true of selection under a budget, and the tables below show both sides of that trade.

**What you will do**

1. Pull candidate traces and their automated scores from Langfuse (LLM-as-a-judge, code checks, user feedback).
2. Declare a daily budget and severity weights for the signals.
3. Compare `random`, `max_score`, `severity`, `uncertainty` and `mixed` selection on the same pool.
4. Create score configs and an annotation queue, and add the selected traces in priority order.
5. Read the annotation scores back and estimate the pushed selection's hit rate and the judge's agreement with humans.

The notebook runs end to end without credentials on a **synthetic** pool with hidden ground truth, so you can read every table before connecting a project. With credentials it reads and writes your Langfuse project. Nothing here measures real-world harm; the synthetic numbers verify the code and are not results.

## Step 1: Install and configure

Leave the keys empty to run offline on synthetic data. Fill them in to run against your project. Region base URLs: `https://cloud.langfuse.com` (EU), `https://us.cloud.langfuse.com` (US), or your self-hosted instance. The reads below use the v3 Scores API (`GET /api/public/v3/scores`), so a self-hosted instance needs a version that serves it.

In [1]:
%pip install langfuse pandas numpy --quiet

In [2]:
import os
from datetime import UTC, datetime, timedelta
from uuid import NAMESPACE_URL, uuid5

import numpy as np
import pandas as pd
from IPython.display import Markdown

os.environ.setdefault("LANGFUSE_PUBLIC_KEY", "")  # "pk-lf-..."
os.environ.setdefault("LANGFUSE_SECRET_KEY", "")  # "sk-lf-..."
os.environ.setdefault("LANGFUSE_BASE_URL", "https://cloud.langfuse.com")

LIVE = bool(os.environ["LANGFUSE_PUBLIC_KEY"] and os.environ["LANGFUSE_SECRET_KEY"])

if LIVE:
    from langfuse import Langfuse

    langfuse = Langfuse()
    assert langfuse.auth_check(), "Langfuse credentials were rejected"

pd.set_option("display.notebook_repr_html", False)


def show(frame: pd.DataFrame, index: bool = True) -> Markdown:
    # A DataFrame as a Markdown table: renders in Jupyter and on the docs site alike.
    table = frame.rename_axis(index=frame.index.name or "").reset_index() if index else frame

    def cell(value) -> str:
        if isinstance(value, float):
            return "" if np.isnan(value) else f"{value:.3f}"
        return str(value)

    header = "| " + " | ".join(str(c) for c in table.columns) + " |"
    rule = "|" + "|".join("---" for _ in table.columns) + "|"
    body = ["| " + " | ".join(cell(v) for v in row) + " |" for row in table.itertuples(index=False)]
    return Markdown("\n".join([header, rule, *body]))


print("mode:", "live" if LIVE else "offline (synthetic pool)")

mode: offline (synthetic pool)


## Step 2: Declare the signals, the budget and the severity weights

Each **signal** is a score name that already exists on your traces: a managed LLM-as-a-judge evaluator, a code evaluator, or a score your own pipeline writes. The notebook reads every signal as a probability-like value in `[0, 1]`; boolean scores arrive as true/false and are read as 1/0. A judge that writes on another scale must be rescaled before it enters the pool, and the pool builder refuses values outside `[0, 1]` rather than clipping them.

The **weights** say how much a positive on each signal is worth to a reviewer's time. They are a policy input you declare, not a measured quantity; every table below is conditional on them. The ratio is what matters: here a likely safety violation outranks a near-certain tone problem.

The **budget** is what your annotators can finish per run. Two annotators, one hour each, three minutes per item, is forty items.

In [3]:
WEIGHTS = {
    "safety_violation": 10.0,  # judge: does the response violate the safety policy?
    "hallucination": 6.0,  # judge: unsupported claims against the retrieved context
    "negative_feedback": 5.0,  # user: thumbs down, written as 1.0
    "off_topic": 2.0,  # judge: response ignores the user's request
    "tone": 1.0,  # judge: tone violates the style guide
}
BUDGET = 40  # items per run
WINDOW_HOURS = 24  # how far back to look for candidates
SEED = 20261006  # for the random policy and the synthetic pool
QUEUE_NAME = "daily-review"
POLICY_SCORE = "review_policy"  # written on each selected trace so reruns skip it

## Step 3: Build the candidate pool

A candidate is a trace with at least one signal in the window that no human has scored, that no earlier run selected (it would carry a `review_policy` score), and that is not already in the queue. The cell also reports two mismatches between `review_policy` records and the queue without acting on them: a decided trace with no queue item (a reviewer removed it, or a push stopped between its two writes) and a queued trace with no decision (added by hand, or decided seconds ago, since score reads lag). Both stay out of the pool. Observation-level judge scores carry their trace id, so the pivot below works whether your evaluators run on traces or on observations; where a trace has several scores of one name, the maximum is kept.

The fetch uses the [v3 scores endpoint](https://api.reference.langfuse.com/#tag/scoresv3) rather than the traces endpoint because every signal, from any source, lands there with its subject. It pages with a cursor and asks for the `subject`, `annotation` and `details` field groups, which add the trace id, the queue id and the metadata. Only the signals are limited to the window. Human annotations and `review_policy` scores are read without a time limit, so a trace a human scored last month stays out when a judge rescores it today; both are small next to the automated scores. Offline, the pool is synthetic: each signal has a hidden true label and a judge score that is informative but imperfect, so the tables can show an oracle column that live data does not have.

In [4]:
SCORE_COLUMNS = ["trace_id", "name", "value", "source", "timestamp", "queue_id", "metadata"]


def trace_id_of(score) -> str | None:
    """The trace a score belongs to; None for session- and experiment-level scores."""
    subject = score.subject
    if subject is None:
        return None
    if subject.kind == "trace":
        return subject.id
    if subject.kind == "observation":
        return subject.trace_id
    return None


def fetch_scores(
    *,
    since: datetime | None = None,
    name: str | None = None,
    source: str | None = None,
    queue_id: str | None = None,
    page_size: int = 100,
) -> pd.DataFrame:
    """Every score matching the filters, one long-form row each.

    `name`, `source` and `queue_id` take comma-separated lists. `value` is
    polymorphic: a number, a boolean, or a string for categorical and text scores.
    """
    rows, cursor = [], None
    while True:
        response = langfuse.api.scores_v3.get_many_v3(
            name=name,
            source=source,
            queue_id=queue_id,
            from_timestamp=since,
            limit=page_size,
            cursor=cursor,
            fields="subject,annotation,details",
        )
        for score in response.data:
            rows.append(
                {
                    "trace_id": trace_id_of(score),
                    "name": score.name,
                    "value": score.value,
                    "source": score.source.value,
                    "timestamp": score.timestamp,
                    "queue_id": score.queue_id,
                    "metadata": score.metadata,
                }
            )
        cursor = response.meta.cursor
        if not cursor:
            break
    return pd.DataFrame(rows, columns=SCORE_COLUMNS)


def queued_trace_ids(queue_id: str, status: str | None = None, page_size: int = 100) -> set[str]:
    """Traces in the queue: every item, or only items with `status` (PENDING or COMPLETED)."""
    ids, page = set(), 1
    while True:
        response = langfuse.api.annotation_queues.list_queue_items(
            queue_id, status=status, page=page, limit=page_size
        )
        ids.update(item.object_id for item in response.data if item.object_type == "TRACE")
        if page >= response.meta.total_pages:
            break
        page += 1
    return ids


def find_queue(name: str):
    page = 1
    while True:
        response = langfuse.api.annotation_queues.list_queues(page=page, limit=100)
        for queue in response.data:
            if queue.name == name:
                return queue
        if page >= response.meta.total_pages:
            return None
        page += 1


def as_unit(value) -> float:
    """Numbers and booleans as floats; strings (categorical, text) as NaN."""
    return float(value) if isinstance(value, bool | int | float) else float("nan")


def build_pool(scores: pd.DataFrame, exclude: set[str]) -> pd.DataFrame:
    """One row per trace, one column per signal; max over duplicate scores; NaN where absent."""
    signals = scores[scores["name"].isin(WEIGHTS) & scores["trace_id"].notna()]
    signals = signals[signals["trace_id"].astype(str) != ""]
    signals = signals[~signals["trace_id"].isin(exclude)]
    signals = signals.assign(value=signals["value"].map(as_unit))
    signals = signals[signals["value"].notna()]  # categorical and text scores are not signals
    out_of_range = sorted(signals.loc[~signals["value"].between(0, 1), "name"].unique())
    if out_of_range:
        raise ValueError(f"rescale these scores into [0, 1] before pooling them: {out_of_range}")
    if signals.empty:
        return pd.DataFrame(columns=["trace_id", *WEIGHTS])
    wide = signals.pivot_table(index="trace_id", columns="name", values="value", aggfunc="max")
    return wide.reindex(columns=list(WEIGHTS)).reset_index().rename_axis(columns=None)

In [5]:
if LIVE:
    since = datetime.now(UTC) - timedelta(hours=WINDOW_HOURS)
    automated = fetch_scores(since=since, name=",".join(WEIGHTS))
    automated = automated[automated["source"] != "ANNOTATION"]
    # No time limit: a trace a human scored before the window must not return as a candidate.
    already_human = set(fetch_scores(source="ANNOTATION")["trace_id"].dropna())
    decisions = fetch_scores(name=POLICY_SCORE)
    already_selected = set(decisions["trace_id"].dropna())
    existing_queue = find_queue(QUEUE_NAME)
    already_queued = queued_trace_ids(existing_queue.id) if existing_queue else set()
    pool = build_pool(automated, already_human | already_selected | already_queued)
    print(
        f"{len(automated)} automated scores on {len(pool)} candidate traces; excluded "
        f"{len(already_human)} annotated, {len(already_selected)} selected earlier, "
        f"{len(already_queued)} already queued"
    )
    if existing_queue:
        # Report only. Step 5 explains why unqueued traces are not added back, and Step 6 why
        # the labels of undecided ones are left out.
        ours = {
            row.trace_id
            for row in decisions.itertuples(index=False)
            if (row.metadata or {}).get("queue_id") == existing_queue.id
        }
        unqueued, undecided = ours - already_queued, already_queued - already_selected
        if unqueued or undecided:
            print(
                f"{len(unqueued)} decided traces are not in the queue (removed by a reviewer, or "
                f"a push that stopped); {len(undecided)} queued traces have no decision (added "
                "by hand, or decided seconds ago)"
            )
    if pool.empty:
        print("No candidates: widen WINDOW_HOURS or check that the signal names match your scores.")
else:
    pool = None  # built from the synthetic generator in the next step

## Step 4: Compare selection policies on the same pool

The cell below is the whole selection logic; copy it into your own pipeline as is. It is kept in sync with [`budget_selection.py`](https://github.com/XinyangWuEthz/review-router/blob/main/examples/langfuse/budget_selection.py) in the review-router repository, where it has a test suite.

| Policy | Ranks by | Mirrors |
|---|---|---|
| `random` | a seeded permutation | sampling, the usual default; not a benchmark arm |
| `max_score` | the highest signal, weights ignored | the benchmark's probability ordering |
| `severity` | the highest of signal times weight over all signals | the benchmark's default ordering |
| `uncertainty` | how close any signal sits to 0.5 | the items a judge is least sure about, which calibrate it; not a benchmark arm |
| `mixed` | quota: 60% severity, 25% uncertainty, 15% random | a hedge: reach the worst, calibrate the judge, keep a drift sample; not a benchmark arm |

A missing signal counts as no evidence, never as evidence. Ties break by trace id so a rerun selects the same rows.

In [6]:
"""Select traces for human annotation under a fixed review budget.

Pure functions over a pandas table with one row per candidate trace and one
column per automated signal (an LLM-as-a-judge score, a code check, a user
feedback flag). Every signal is read as a probability-like value in [0, 1];
a missing signal counts as no evidence (0), never as evidence.

Two policies mirror arms of the review-router benchmark: ``max_score`` is
its probability ordering and ``severity`` its default, the maximum over
signals of probability times a declared weight. ``random`` is this module's
sampling baseline, ``uncertainty`` prefers signals near 0.5 for judge
calibration, and ``mixed`` splits the budget between them by quota; these
three were not benchmark arms. Selecting within a budget reallocates fixed
reviewer capacity; it does not add capacity, and nothing here measures
real-world harm.

This file is inlined verbatim into the companion notebook; the test suite
keeps the two copies identical.
"""

from collections.abc import Iterable, Mapping
from dataclasses import dataclass, field
from math import isfinite, sqrt

import numpy as np
import pandas as pd

__all__ = [
    "POLICIES",
    "FLAG_THRESHOLD",
    "RESERVED_COLUMNS",
    "SelectionConfig",
    "severity_scores",
    "max_scores",
    "uncertainty_scores",
    "policy_scores",
    "rank",
    "select",
    "select_all",
    "compare_policies",
    "overlap_matrix",
    "synthetic_pool",
    "simulate_annotations",
    "annotation_precision",
    "judge_agreement",
]

POLICIES: tuple[str, ...] = ("random", "max_score", "severity", "uncertainty", "mixed")
# A signal at or above this value counts as "flagged" in the composition tables.
FLAG_THRESHOLD = 0.5
_SCORED = ("max_score", "severity", "uncertainty")
# Columns rank() and select() add; a signal with one of these names would be overwritten.
RESERVED_COLUMNS = frozenset({"policy", "score", "rank", "source", "severity_score"})


@dataclass(frozen=True)
class SelectionConfig:
    """Budget, severity weights and the quota split of the mixed policy."""

    budget: int
    weights: Mapping[str, float]
    seed: int = 0
    mixed_quota: Mapping[str, float] = field(
        default_factory=lambda: {"severity": 0.6, "uncertainty": 0.25, "random": 0.15}
    )

    def __post_init__(self) -> None:
        if (
            isinstance(self.budget, bool)
            or not isinstance(self.budget, int | np.integer)
            or self.budget <= 0
        ):
            raise ValueError("budget must be a positive integer")
        if not self.weights:
            raise ValueError("weights must name at least one signal")
        for name, weight in self.weights.items():
            if (
                isinstance(weight, bool)
                or not isinstance(weight, int | float | np.integer | np.floating)
                or not isfinite(weight)
                or weight <= 0
            ):
                raise ValueError(f"weight for {name!r} must be positive and finite")
            if name in RESERVED_COLUMNS:
                raise ValueError(f"signal name {name!r} is reserved; rename the score")
        unknown = set(self.mixed_quota) - {*_SCORED, "random"}
        if unknown:
            raise ValueError(f"mixed_quota names unknown policies: {sorted(unknown)}")
        for share in self.mixed_quota.values():
            if (
                isinstance(share, bool)
                or not isinstance(share, int | float | np.integer | np.floating)
                or not isfinite(share)
                or share < 0
            ):
                raise ValueError("mixed_quota shares must be finite and nonnegative")
        if abs(sum(self.mixed_quota.values()) - 1.0) > 1e-9:
            raise ValueError("mixed_quota shares must sum to 1")

    @property
    def signals(self) -> list[str]:
        return list(self.weights)


def _signal_matrix(pool: pd.DataFrame, weights: Mapping[str, float]) -> np.ndarray:
    missing = [name for name in weights if name not in pool.columns]
    if missing:
        raise ValueError(f"pool lacks signal columns: {missing}")
    values = pool[list(weights)].to_numpy(dtype=float)
    if np.nanmin(values, initial=0.0) < 0 or np.nanmax(values, initial=0.0) > 1:
        raise ValueError("signals must lie in [0, 1]")
    return np.asarray(np.nan_to_num(values, nan=0.0), dtype=float)


def severity_scores(pool: pd.DataFrame, weights: Mapping[str, float]) -> pd.Series:
    """Maximum over signals of probability times weight; the review-router default."""
    p = _signal_matrix(pool, weights)
    w = np.array([weights[name] for name in weights], dtype=float)
    return pd.Series((p * w).max(axis=1) if len(p) else [], index=pool.index, dtype=float)


def max_scores(pool: pd.DataFrame, weights: Mapping[str, float]) -> pd.Series:
    """Maximum probability over signals, ignoring the weights."""
    p = _signal_matrix(pool, weights)
    return pd.Series(p.max(axis=1) if len(p) else [], index=pool.index, dtype=float)


def uncertainty_scores(pool: pd.DataFrame, weights: Mapping[str, float]) -> pd.Series:
    """1 at a signal of 0.5, 0 at 0 or 1; highest where some judge is least sure."""
    p = _signal_matrix(pool, weights)
    closeness = 1.0 - np.abs(2.0 * p - 1.0)
    return pd.Series(closeness.max(axis=1) if len(p) else [], index=pool.index, dtype=float)


def policy_scores(pool: pd.DataFrame, policy: str, config: SelectionConfig) -> pd.Series:
    """The ranking score of one policy. random draws a seeded permutation."""
    if policy == "severity":
        return severity_scores(pool, config.weights)
    if policy == "max_score":
        return max_scores(pool, config.weights)
    if policy == "uncertainty":
        return uncertainty_scores(pool, config.weights)
    if policy == "random":
        rng = np.random.default_rng([config.seed, len(pool)])
        return pd.Series(rng.random(len(pool)), index=pool.index, dtype=float)
    raise ValueError(f"unknown policy {policy!r}; mixed has no single score")


def _check_pool(pool: pd.DataFrame) -> None:
    if "trace_id" not in pool.columns:
        raise ValueError("pool needs a trace_id column")
    if pool["trace_id"].isna().any() or (pool["trace_id"].astype(str) == "").any():
        raise ValueError("every candidate needs a non-empty trace_id")
    if pool["trace_id"].duplicated().any():
        raise ValueError("trace_id values must be unique; aggregate signals per trace first")
    clash = sorted(RESERVED_COLUMNS.intersection(pool.columns))
    if clash:
        raise ValueError(f"pool columns {clash} are reserved for the selection output; rename them")


def rank(pool: pd.DataFrame, policy: str, config: SelectionConfig) -> pd.DataFrame:
    """The whole pool ordered by one policy: score descending, ties by trace_id.

    ``score`` is the policy's own ranking score; ``severity_score`` is added to
    every row so selections from different policies share one comparable number.
    """
    _check_pool(pool)
    if policy == "mixed":
        raise ValueError("mixed selects by quota; use select()")
    scored = pool.assign(
        policy=policy,
        score=policy_scores(pool, policy, config),
        severity_score=severity_scores(pool, config.weights),
    )
    ordered = scored.sort_values(["score", "trace_id"], ascending=[False, True], kind="stable")
    return ordered.assign(rank=np.arange(1, len(ordered) + 1)).reset_index(drop=True)


def _quota_counts(budget: int, quota: Mapping[str, float]) -> dict[str, int]:
    """Largest-remainder split of the budget; the counts always sum to the budget."""
    names = list(quota)
    exact = np.array([budget * quota[name] for name in names], dtype=float)
    counts = np.floor(exact).astype(int)
    for i in np.argsort(-(exact - counts), kind="stable")[: budget - int(counts.sum())]:
        counts[i] += 1
    return dict(zip(names, (int(c) for c in counts), strict=True))


def select(pool: pd.DataFrame, policy: str, config: SelectionConfig) -> pd.DataFrame:
    """The rows one policy sends to review, at most the budget, in service order.

    Columns: trace_id, policy, score, severity_score, rank, source (the
    sub-policy that took the row; equals policy except under mixed), plus the
    pool's own columns. Under mixed, ``score`` is the taking sub-policy's own
    score and is only comparable within one ``source``; ``severity_score`` is
    comparable across rows.
    """
    _check_pool(pool)
    if policy != "mixed":
        chosen = rank(pool, policy, config).head(config.budget)
        return chosen.assign(source=policy)
    taken: list[pd.DataFrame] = []
    seen: set[str] = set()
    for name, count in _quota_counts(config.budget, config.mixed_quota).items():
        ordered = rank(pool, name, config)
        fresh = ordered[~ordered["trace_id"].isin(seen)].head(count)
        seen.update(fresh["trace_id"])
        taken.append(fresh.assign(source=name))
    # Quotas can be starved by overlap; refill from severity so the budget is spent.
    shortfall = config.budget - sum(len(part) for part in taken)
    if shortfall > 0:
        ordered = rank(pool, "severity", config)
        fill = ordered[~ordered["trace_id"].isin(seen)].head(shortfall)
        taken.append(fill.assign(source="severity"))
    chosen = (
        pd.concat(taken, ignore_index=True) if taken else rank(pool, "severity", config).head(0)
    )
    return chosen.assign(policy="mixed", rank=np.arange(1, len(chosen) + 1))


def select_all(
    pool: pd.DataFrame, config: SelectionConfig, policies: Iterable[str] = POLICIES
) -> dict[str, pd.DataFrame]:
    return {policy: select(pool, policy, config) for policy in _names(policies)}


def _truth_label(value: object) -> float:
    if isinstance(value, str):
        return {"true": 1.0, "false": 0.0, "1": 1.0, "0": 0.0}.get(value.strip().lower(), 2.0)
    try:
        return float(value)  # type: ignore[arg-type]
    except (TypeError, ValueError):
        return 2.0


def _truth_mask(column: pd.Series, name: str) -> np.ndarray:
    """0/1, False/True or their strings as a boolean mask; NaN counts as not positive."""
    labels = (
        column.astype(object).where(column.notna(), 0.0).map(_truth_label).to_numpy(dtype=float)
    )
    if not np.isin(labels, (0.0, 1.0)).all():
        raise ValueError(f"truth column {name!r} must hold 0/1 or False/True values")
    return np.asarray(labels == 1.0)


def compare_policies(
    pool: pd.DataFrame,
    config: SelectionConfig,
    policies: Iterable[str] = POLICIES,
    truth: str | None = None,
) -> pd.DataFrame:
    """What each policy's selection contains, and what it leaves out.

    Per policy: rows selected, rows flagged (any signal >= 0.5) among them,
    per-signal flagged counts, mean severity of the selection, and
    ``flagged_left_out``: flagged rows that did not fit in the budget. With a
    boolean ``truth`` column (an oracle, only available on synthetic or
    already-labeled data; 0/1 or False/True, NaN meaning unlabeled and
    counted as not positive) it adds the true positives captured and the
    share of all true positives in the pool that the budget reaches.
    """
    _check_pool(pool)
    positives = np.zeros(len(pool), dtype=bool)
    if truth is not None:
        positives = _truth_mask(pool[truth], truth)
    flags = _signal_matrix(pool, config.weights) >= FLAG_THRESHOLD
    any_flag = pd.Series(flags.any(axis=1), index=pool.index)
    severity = severity_scores(pool, config.weights)
    rows = []
    for policy in _names(policies):
        chosen = select(pool, policy, config)
        mask = pool["trace_id"].isin(chosen["trace_id"]).to_numpy()
        row: dict[str, object] = {
            "policy": policy,
            "selected": int(mask.sum()),
            "flagged_selected": int(any_flag[mask].sum()),
            "flagged_left_out": int(any_flag[~mask].sum()),
            "mean_severity": float(severity[mask].mean()) if mask.any() else 0.0,
        }
        for j, name in enumerate(config.signals):
            row[f"{name}_flagged"] = int(flags[mask, j].sum())
        if truth is not None:
            captured = int((positives & mask).sum())
            row["true_positive_selected"] = captured
            row["true_positive_share"] = (
                captured / int(positives.sum()) if positives.any() else float("nan")
            )
        rows.append(row)
    return pd.DataFrame(rows)


def overlap_matrix(selections: Mapping[str, pd.DataFrame]) -> pd.DataFrame:
    """Count of trace_ids two policies both selected."""
    ids = {name: set(frame["trace_id"]) for name, frame in selections.items()}
    names = list(ids)
    matrix = pd.DataFrame(
        [[len(ids[a] & ids[b]) for b in names] for a in names], index=names, columns=names
    )
    return matrix.rename_axis(index="policy")


def synthetic_pool(
    n: int,
    weights: Mapping[str, float],
    seed: int = 0,
    prevalence: Mapping[str, float] | None = None,
    missing_rate: float = 0.05,
) -> pd.DataFrame:
    """SYNTHETIC candidates with hidden truth, for running the notebook offline.

    Each signal has a latent true label drawn at its prevalence (default 0.08)
    and a judge score drawn from Beta(5, 2) when the label is 1 and Beta(1, 6)
    when it is 0, so the judge is informative but imperfect. A share of
    signals is missing (NaN). ``true_<signal>`` columns hold the latent labels
    and ``true_high_risk`` is 1 when the maximum true-label weight reaches the
    median declared weight. Numbers from this pool verify the code; they are
    not results.
    """
    if n <= 0:
        raise ValueError("n must be positive")
    if not 0 <= missing_rate < 1:
        raise ValueError("missing_rate must lie in [0, 1)")
    rng = np.random.default_rng(seed)
    prevalence = dict(prevalence or {})
    frame = pd.DataFrame({"trace_id": [f"trace-{seed}-{i:05d}" for i in range(n)]})
    truth_weight = np.zeros(n)
    for name, weight in weights.items():
        y = rng.random(n) < prevalence.get(name, 0.08)
        p = np.where(y, rng.beta(5.0, 2.0, size=n), rng.beta(1.0, 6.0, size=n))
        p[rng.random(n) < missing_rate] = np.nan
        frame[name] = p
        frame[f"true_{name}"] = y.astype(int)
        truth_weight = np.maximum(truth_weight, y * float(weight))
    frame["true_high_risk"] = (truth_weight >= float(np.median(list(weights.values())))).astype(int)
    return frame


def simulate_annotations(
    selected: pd.DataFrame,
    pool: pd.DataFrame,
    signals: Iterable[str],
    seed: int = 0,
    flip: float = 0.1,
) -> pd.DataFrame:
    """SYNTHETIC annotator labels for selected rows: hidden truth with a flip rate.

    Returns long-form rows (trace_id, name, value) in the shape the Langfuse
    scores endpoint returns for ANNOTATION-source scores, so the closing-the-
    loop cells run identically offline and live.
    """
    rng = np.random.default_rng(seed)
    truth = pool.set_index("trace_id")
    names = _names(signals)
    rows = []
    for trace_id in selected["trace_id"].drop_duplicates():
        for name in names:
            label = int(truth.at[trace_id, f"true_{name}"])
            if rng.random() < flip:
                label = 1 - label
            rows.append({"trace_id": trace_id, "name": f"human_{name}", "value": float(label)})
    return pd.DataFrame(rows, columns=["trace_id", "name", "value"])


def _names(values: Iterable[str]) -> list[str]:
    """Materialize a name list; a bare string is a mistake, not a one-item list."""
    if isinstance(values, str):
        raise TypeError("pass an iterable of names, not a single string")
    return list(values)


def _wilson(successes: int, n: int, z: float = 1.959964) -> tuple[float, float]:
    if n == 0:
        return (float("nan"), float("nan"))
    p = successes / n
    denominator = 1 + z * z / n
    center = (p + z * z / (2 * n)) / denominator
    half = z * sqrt(p * (1 - p) / n + z * z / (4 * n * n)) / denominator
    return (max(0.0, center - half), min(1.0, center + half))


def annotation_precision(
    selections: Mapping[str, pd.DataFrame], annotations: pd.DataFrame, positive_names: Iterable[str]
) -> pd.DataFrame:
    """Share of each policy's annotated rows that a human marked positive on any listed score.

    ``annotations`` is long-form (trace_id, name, value) with value 1 for a
    positive label. Rows a human has not labeled yet are excluded from the
    denominator, so early in a review cycle the estimate covers only the
    items reviewers reached. Intervals are 95% Wilson.
    """
    names = set(_names(positive_names))
    labeled = annotations[annotations["name"].isin(names)]
    positive_ids = set(labeled.loc[labeled["value"] >= 1, "trace_id"])
    labeled_ids = set(labeled["trace_id"])
    rows = []
    for policy, chosen in selections.items():
        ids = [t for t in chosen["trace_id"] if t in labeled_ids]
        hits = sum(t in positive_ids for t in ids)
        low, high = _wilson(hits, len(ids))
        rows.append(
            {
                "policy": policy,
                "annotated": len(ids),
                "positive": hits,
                "precision": hits / len(ids) if ids else float("nan"),
                "ci_low": low,
                "ci_high": high,
            }
        )
    columns = ["policy", "annotated", "positive", "precision", "ci_low", "ci_high"]
    return pd.DataFrame(rows, columns=columns)


def judge_agreement(
    pool: pd.DataFrame, annotations: pd.DataFrame, signals: Iterable[str]
) -> pd.DataFrame:
    """Per signal, how the judge flag (>= 0.5) agrees with the human label on annotated rows.

    A row the judge did not score holds no prediction: it is left out of
    ``annotated`` and the rates, and counted in ``judge_missing``. Rows were
    chosen by the policies, not at random, so these rates describe the
    reviewed slice only; they are not the judge's accuracy on all traffic.
    """
    scores = pool.set_index("trace_id")
    rows = []
    for name in _names(signals):
        human = annotations[annotations["name"] == f"human_{name}"]
        human = human[human["trace_id"].isin(scores.index)]
        judge = scores.loc[human["trace_id"], name].to_numpy(dtype=float)
        scored = ~np.isnan(judge)
        judge_flag = judge[scored] >= FLAG_THRESHOLD
        label = human["value"].to_numpy(dtype=float)[scored] >= 1
        both = int((judge_flag & label).sum())
        rows.append(
            {
                "signal": name,
                "annotated": int(scored.sum()),
                "judge_missing": int((~scored).sum()),
                "agreement": float((judge_flag == label).mean()) if scored.any() else float("nan"),
                "judge_precision": both / int(judge_flag.sum())
                if judge_flag.any()
                else float("nan"),
                "judge_recall": both / int(label.sum()) if label.any() else float("nan"),
            }
        )
    columns = [
        "signal",
        "annotated",
        "judge_missing",
        "agreement",
        "judge_precision",
        "judge_recall",
    ]
    return pd.DataFrame(rows, columns=columns)

In [7]:
if not LIVE:
    pool = synthetic_pool(2000, WEIGHTS, seed=SEED)

config = SelectionConfig(budget=BUDGET, weights=WEIGHTS, seed=SEED)
truth = None if LIVE else "true_high_risk"

print(f"{len(pool)} candidates, budget {BUDGET}")
show(pool[["trace_id", *WEIGHTS]].head(), index=False)

2000 candidates, budget 40


| trace_id | safety_violation | hallucination | negative_feedback | off_topic | tone |
|---|---|---|---|---|---|
| trace-20261006-00000 |  | 0.144 | 0.116 | 0.108 | 0.020 |
| trace-20261006-00001 | 0.125 | 0.011 | 0.108 | 0.003 | 0.042 |
| trace-20261006-00002 | 0.100 | 0.028 | 0.021 | 0.075 | 0.550 |
| trace-20261006-00003 | 0.224 | 0.048 | 0.180 | 0.006 |  |
| trace-20261006-00004 |  | 0.205 |  | 0.148 | 0.035 |

### What each policy's budget contains, and what it leaves out

Read the table in pairs. `flagged_selected` is how many selected rows have any signal at or above 0.5; `flagged_left_out` is how many flagged rows did not fit in the budget. That second number is the displaced work: it is the cost of every policy, and it is large whenever the pool is large. The per-signal columns show *which* flags each policy spends the budget on. Offline, `true_positive_share` uses the hidden truth and is the share of all truly high-risk rows in the pool that the budget reaches; live data has no such column until humans have scored it.

In [8]:
comparison = compare_policies(pool, config, truth=truth)
show(comparison.set_index("policy"))

| policy | selected | flagged_selected | flagged_left_out | mean_severity | safety_violation_flagged | hallucination_flagged | negative_feedback_flagged | off_topic_flagged | tone_flagged | true_positive_selected | true_positive_share |
|---|---|---|---|---|---|---|---|---|---|---|---|
| random | 40 | 18 | 686 | 2.945 | 5 | 3 | 5 | 5 | 2 | 13 | 0.029 |
| max_score | 40 | 40 | 664 | 5.262 | 9 | 11 | 12 | 12 | 9 | 29 | 0.064 |
| severity | 40 | 40 | 664 | 9.119 | 40 | 1 | 1 | 0 | 5 | 40 | 0.088 |
| uncertainty | 40 | 31 | 673 | 3.331 | 5 | 10 | 10 | 11 | 6 | 13 | 0.029 |
| mixed | 40 | 34 | 670 | 6.713 | 24 | 4 | 6 | 2 | 7 | 27 | 0.060 |

### How much the policies agree

Two policies that choose the same rows cost the same and teach the same. The matrix counts shared traces. How much `severity` and `max_score` overlap depends on how spread the weights are: with a 10:1 spread they share few rows; with near-flat weights they converge. `uncertainty` and `random` mostly reach rows the ranked policies do not.

In [9]:
selections = select_all(pool, config)
show(overlap_matrix(selections))

| policy | random | max_score | severity | uncertainty | mixed |
|---|---|---|---|---|---|
| random | 40 | 0 | 3 | 1 | 8 |
| max_score | 0 | 40 | 9 | 1 | 9 |
| severity | 3 | 9 | 40 | 0 | 24 |
| uncertainty | 1 | 1 | 0 | 40 | 10 |
| mixed | 8 | 9 | 24 | 10 | 40 |

In [10]:
# The mixed selection in the order the queue will serve it. `source` is the quota that took the
# row, `score` that quota's own ranking score, `severity_score` the number comparable across rows.
selection = selections["mixed"]
show(
    selection[["rank", "source", "score", "severity_score", "trace_id", *WEIGHTS]].head(12),
    index=False,
)

| rank | source | score | severity_score | trace_id | safety_violation | hallucination | negative_feedback | off_topic | tone |
|---|---|---|---|---|---|---|---|---|---|
| 1 | severity | 9.760 | 9.760 | trace-20261006-01920 | 0.976 | 0.112 | 0.290 | 0.130 | 0.756 |
| 2 | severity | 9.728 | 9.728 | trace-20261006-01786 | 0.973 | 0.014 | 0.648 |  | 0.069 |
| 3 | severity | 9.707 | 9.707 | trace-20261006-00260 | 0.971 | 0.106 | 0.024 | 0.280 | 0.108 |
| 4 | severity | 9.664 | 9.664 | trace-20261006-01925 | 0.966 | 0.037 | 0.386 | 0.093 | 0.239 |
| 5 | severity | 9.651 | 9.651 | trace-20261006-00035 | 0.965 | 0.257 | 0.468 | 0.149 | 0.900 |
| 6 | severity | 9.630 | 9.630 | trace-20261006-00664 | 0.963 | 0.209 | 0.031 | 0.464 | 0.081 |
| 7 | severity | 9.560 | 9.560 | trace-20261006-01573 | 0.956 | 0.021 | 0.488 | 0.001 | 0.081 |
| 8 | severity | 9.482 | 9.482 | trace-20261006-01348 | 0.948 | 0.108 | 0.109 | 0.074 | 0.170 |
| 9 | severity | 9.397 | 9.397 | trace-20261006-00190 | 0.940 | 0.098 | 0.026 | 0.320 | 0.012 |
| 10 | severity | 9.379 | 9.379 | trace-20261006-00760 | 0.938 | 0.024 | 0.191 | 0.014 | 0.273 |
| 11 | severity | 9.317 | 9.317 | trace-20261006-00271 | 0.932 | 0.169 | 0.035 | 0.201 |  |
| 12 | severity | 9.311 | 9.311 | trace-20261006-00196 | 0.931 | 0.442 | 0.008 | 0.025 | 0.901 |

## Step 5: Push the selection to an annotation queue

Four API calls do the work: ensure a boolean [score config](https://api.reference.langfuse.com/#tag/scoreconfigs) per human label, ensure a queue that offers those configs, and, for each selected trace, record the decision and add the queue item. Both ensure steps reuse what already exists, and they stop with an error when an existing config is not boolean or an existing queue lacks the configs, because reviewers could not then give the labels Step 6 reads. Score config names are limited to 35 characters and to letters, digits, underscores, spaces, periods, parentheses and hyphens, so the `human_` prefix leaves 29 characters for a signal name. Three details matter.

**Insertion order is the priority.** A queue item has no priority field. Annotators are served pending items oldest first, so inserting in rank order is what sets the priority; the items endpoint itself lists newest first. Check the served order once in your own project after the first run; if it differs, split the selection into one queue per tier instead.

**The API does not reject duplicates.** Adding a trace that is already in the queue creates a second item, so `push_selection` lists the queue's items first and skips known ids. That check, together with the `review_policy` exclusion in Step 3, is what keeps a trace from being queued twice.

**Record the decision first.** Each selected trace receives a categorical score named `review_policy` whose value is the quota that selected it, with metadata naming the queue, every policy that would also have chosen it, and the signal values it was chosen on. Step 6 reads that record back, so the human labels can be attributed to policies and compared with what the judges said at selection time, long after this run's memory is gone. The record goes through the synchronous scores endpoint, before the queue item, because `create_score` batches in the background, never raises on a failed send, and drops scores of unsampled traces when `LANGFUSE_SAMPLE_RATE` is below 1; it cannot promise that a queued trace has its record. If the cell stops partway, rerun it. It resumes where it stopped: a decision this session already wrote is not written again, and an item a reviewer removed in the meantime stays removed. The score id is derived from the trace id, but Langfuse replaces a score with the same id only within the same UTC day, so a decision repeated on a later day leaves a second record; Step 6 keeps the latest. After a kernel restart, a record without its item is reported in Step 3 and not added back, because it looks the same as an item a reviewer removed. Add the trace by hand to review it anyway.

In [11]:
HUMAN_SCORES = {f"human_{name}": f"Reviewer confirms: {name.replace('_', ' ')}" for name in WEIGHTS}
# A reviewed row counts as a hit when a human confirms a high-weight problem.
POSITIVE_LABELS = [f"human_{name}" for name, weight in WEIGHTS.items() if weight >= 5]
assert all(len(name) <= 35 for name in HUMAN_SCORES), (
    "score config names are limited to 35 characters"
)


def ensure_score_config(name: str, description: str):
    page = 1
    while True:
        response = langfuse.api.score_configs.get(page=page, limit=100)
        for cfg in response.data:
            if cfg.name == name and not cfg.is_archived:
                if cfg.data_type.value != "BOOLEAN":
                    raise ValueError(
                        f"score config {name!r} exists as {cfg.data_type.value}; Step 6 reads "
                        "true/false labels, so archive it or rename the signal"
                    )
                return cfg
        if page >= response.meta.total_pages:
            break
        page += 1
    return langfuse.api.score_configs.create(
        name=name, data_type="BOOLEAN", description=description
    )


def ensure_queue(name: str, score_config_ids: list[str]):
    queue = find_queue(name)
    if queue is None:
        return langfuse.api.annotation_queues.create_queue(
            name=name,
            score_config_ids=score_config_ids,
            description="Daily human review, selected under a fixed budget",
        )
    if not set(score_config_ids) <= set(queue.score_config_ids):
        raise ValueError(
            f"queue {name!r} exists without the human_* score configs, so reviewers could not "
            "give the labels Step 6 reads; set QUEUE_NAME to a new name"
        )
    return queue


def decision_score_id(trace_id: str) -> str:
    """The review_policy score's id, fixed per trace: a retry on the same day replaces it."""
    return str(uuid5(NAMESPACE_URL, f"{POLICY_SCORE}/{trace_id}"))


# What this session has written. A rerun of Step 5 resumes where it stopped: it never writes a
# decision twice, and never puts back an item a reviewer removed in the meantime.
decided_this_session: set[str] = set()
queued_this_session: set[str] = set()


def push_selection(selections: dict[str, pd.DataFrame], pushed: str, queue_id: str) -> int:
    """Record each decision, then add the trace to the queue, in rank order.

    The decision goes through the synchronous scores endpoint, which raises on failure, so no
    trace enters the queue without one. Rerunning after a failure resumes where it stopped and
    skips the writes this session already made.
    """
    selection = selections[pushed]
    membership = {policy: set(frame["trace_id"]) for policy, frame in selections.items()}
    signals = selection.set_index("trace_id")[list(WEIGHTS)]
    present = queued_trace_ids(queue_id)
    added = 0
    for row in selection.sort_values("rank").itertuples(index=False):
        if row.trace_id in queued_this_session:
            continue
        if row.trace_id not in decided_this_session:
            langfuse.api.scores.create(
                id=decision_score_id(row.trace_id),
                trace_id=row.trace_id,
                name=POLICY_SCORE,
                value=row.source,
                data_type="CATEGORICAL",
                comment=f"rank {row.rank} of {len(selection)}; severity {row.severity_score:.2f}",
                metadata={
                    "queue_id": queue_id,
                    "pushed": pushed,
                    "rank": int(row.rank),
                    "severity_score": float(row.severity_score),
                    "policies": [p for p, ids in membership.items() if row.trace_id in ids],
                    # The signals as selection saw them; Step 6 compares them with the labels.
                    "signals": {
                        name: None if pd.isna(value) else float(value)
                        for name, value in signals.loc[row.trace_id].items()
                    },
                },
            )
            decided_this_session.add(row.trace_id)
        if row.trace_id not in present:
            langfuse.api.annotation_queues.create_queue_item(
                queue_id, object_id=row.trace_id, object_type="TRACE"
            )
            added += 1
        queued_this_session.add(row.trace_id)
    return added

In [12]:
if LIVE:
    configs = [ensure_score_config(name, text) for name, text in HUMAN_SCORES.items()]
    queue = ensure_queue(QUEUE_NAME, [cfg.id for cfg in configs])
    added = push_selection(selections, "mixed", queue.id)
    print(f"queue {queue.name!r} ({queue.id}): added {added} of {len(selection)} selected traces")
    # Optional: route the queue to specific annotators.
    # langfuse.api.annotation_queues.create_queue_assignment(queue.id, user_id="<langfuse user id>")
else:
    print(
        f"offline: would create {len(HUMAN_SCORES)} boolean score configs, the queue "
        f"{QUEUE_NAME!r}, and add {len(selection)} items in this order:"
    )
    display(show(selection[["rank", "source", "severity_score", "trace_id"]].head(5), index=False))

offline: would create 5 boolean score configs, the queue 'daily-review', and add 40 items in this order:


| rank | source | severity_score | trace_id |
|---|---|---|---|
| 1 | severity | 9.760 | trace-20261006-01920 |
| 2 | severity | 9.728 | trace-20261006-01786 |
| 3 | severity | 9.707 | trace-20261006-00260 |
| 4 | severity | 9.664 | trace-20261006-01925 |
| 5 | severity | 9.651 | trace-20261006-00035 |

## Step 6: Close the loop with the human scores

Once annotators have worked the queue, every label is a score with `source = ANNOTATION` and the queue's id. Live, this step runs on a later day than Step 5, so it does not use this run's memory. It reads the labels on queue items marked completed, so a review still in progress does not count as a miss. It also reads the `review_policy` scores back, which record the policies that chose each reviewed trace and the signals the trace had when they chose it. A reviewed trace without a record (added by hand, or queued by an earlier version whose background write was lost) is counted and left out of both tables, since no policy is known to have chosen it. Reading them back answers two questions.

**Which policy's picks did humans confirm?** `annotation_precision` counts, per policy, the share of its annotated rows that a human marked positive on a high-weight label (`POSITIVE_LABELS`, the signals weighted 5 or more). Offline, every policy's selection is labeled, so the table compares all five. Live, only the pushed selection is labeled. Its row covers the whole selection. Every other row covers only the traces that policy shares with the pushed one, and those can sit anywhere in its own ranking, so the row describes the shared traces and not the policy. Ranking the policies against each other on live data needs labels on each policy's full selection, which a queue filled by one policy does not collect.

**How well does the judge agree with humans?** `judge_agreement` compares each signal's flag at 0.5, as it stood when the trace was selected, with the human label on the annotated rows. A trace the judge never scored has no prediction to compare; it is left out and counted in `judge_missing`. The annotated rows were chosen by the policies, not at random, so the numbers describe the reviewed slice and not the judge's accuracy on all traffic. `judge_precision` on the flagged rows the budget reached is a fair check for false alarms at the top of the ranking. `judge_recall` on this slice is biased upward, because positives the judge scored low rarely enter it; only the random quota gives it meaning, so treat a low value as a warning and a high value as uninformative.

In [13]:
if LIVE:
    queue = find_queue(QUEUE_NAME)  # looked up again, so this step can run on a later day
    assert queue is not None, f"no queue named {QUEUE_NAME!r}; run Step 5 first"
    # Completed items only: a label in progress can change, and an unanswered field is not a no.
    completed = queued_trace_ids(queue.id, status="COMPLETED")
    annotations = fetch_scores(source="ANNOTATION", queue_id=queue.id)
    annotations = annotations[
        annotations["name"].isin(HUMAN_SCORES) & annotations["trace_id"].isin(completed)
    ]
    annotations = annotations.assign(value=annotations["value"].map(as_unit))
    annotations = annotations[["trace_id", "name", "value"]].dropna()
    reviewed_ids = set(annotations["trace_id"])

    # From the stored decision: the policies that chose each reviewed trace, and the signals
    # the trace had when they chose it.
    decisions = fetch_scores(name=POLICY_SCORE)
    decisions = decisions[decisions["trace_id"].isin(reviewed_ids)]
    # A decision repeated on a later day is a second record; the latest one wins.
    decisions = decisions.sort_values("timestamp", ascending=False).drop_duplicates("trace_id")
    unattributed = reviewed_ids - set(decisions["trace_id"])
    if unattributed:
        print(f"{len(unattributed)} reviewed traces have no review_policy record; left out below")
    chosen_by: dict[str, list[str]] = {}
    at_selection = []
    for row in decisions.itertuples(index=False):
        metadata = row.metadata or {}
        for policy in metadata.get("policies") or [row.value]:
            chosen_by.setdefault(policy, []).append(row.trace_id)
        at_selection.append({"trace_id": row.trace_id, **(metadata.get("signals") or {})})
    reviewed_selections = {p: pd.DataFrame({"trace_id": ids}) for p, ids in chosen_by.items()}
    reviewed_pool = pd.DataFrame(at_selection, columns=["trace_id", *WEIGHTS])
    reviewed_pool = reviewed_pool.astype({name: float for name in WEIGHTS})
else:
    # SYNTHETIC: label every policy's selection from the hidden truth, with a 10% flip rate.
    union = pd.concat(selections.values(), ignore_index=True).drop_duplicates("trace_id")
    annotations = simulate_annotations(union, pool, WEIGHTS, seed=SEED, flip=0.1)
    reviewed_selections, reviewed_pool = selections, pool

if LIVE and not reviewed_ids:
    print(f"no completed reviews in queue {queue.name!r} yet; rerun Step 6 once some are done")
print(f"{len(annotations)} human labels on {annotations['trace_id'].nunique()} traces")
show(annotation_precision(reviewed_selections, annotations, POSITIVE_LABELS).set_index("policy"))

730 human labels on 146 traces


| policy | annotated | positive | precision | ci_low | ci_high |
|---|---|---|---|---|---|
| random | 40 | 22 | 0.550 | 0.398 | 0.693 |
| max_score | 40 | 32 | 0.800 | 0.652 | 0.895 |
| severity | 40 | 33 | 0.825 | 0.681 | 0.913 |
| uncertainty | 40 | 21 | 0.525 | 0.375 | 0.671 |
| mixed | 40 | 26 | 0.650 | 0.495 | 0.779 |

In [14]:
show(judge_agreement(reviewed_pool, annotations, WEIGHTS).set_index("signal"))

| signal | annotated | judge_missing | agreement | judge_precision | judge_recall |
|---|---|---|---|---|---|
| safety_violation | 143 | 3 | 0.825 | 0.787 | 0.712 |
| hallucination | 140 | 6 | 0.821 | 0.600 | 0.500 |
| negative_feedback | 138 | 8 | 0.870 | 0.630 | 0.680 |
| off_topic | 138 | 8 | 0.819 | 0.630 | 0.531 |
| tone | 142 | 4 | 0.880 | 0.684 | 0.542 |

## Step 7: Run it on a schedule

Schedule the notebook, or the module and the fetch helpers, once per review cycle. The `review_policy` score and the queue membership check keep a trace from being selected or queued twice, but every run pushes a fresh budget of new traces. If a scheduled run fails partway, rerun only Step 5 in the same session, or lower `BUDGET` for the retry by the items the failed run queued. After a few cycles, revisit the inputs with the two tables above:

- Track the pushed policy's precision from cycle to cycle. If it falls, either the traffic has fewer problems or the signals have drifted, and a falling `judge_precision` points to the second. The other policies' rows cover only their overlap with the pushed selection, so do not set quotas from them.
- A signal whose `judge_precision` is low is pulling rows into the budget on false alarms; fix the judge prompt before raising its weight.
- A signal whose `judge_recall` is low on the reviewed slice is one the random quota exists for; raise that quota if you suspect drift. A high value on this slice says nothing.

## What this establishes, and what it does not

Selection under a budget reallocates reviewer time. In the review-router benchmark, at 180 admitted jobs per hour against capacity for 120, the severity ordering completed 53.75 more high-risk reviews and 53.75 fewer other reviews per eight-hour shift than first-in-first-out, averaged over 20 paired simulation seeds, with the total unchanged. The [experiment record](https://xinyangwuethz.github.io/review-router/), the [weight sensitivity check](https://github.com/XinyangWuEthz/review-router/blob/main/record/severity-sensitivity.md) that holds those numbers, and the article [Same model, different review queue](https://xinyangwuethz.github.io/notes/same-model-different-review-queue/) walk through that result and its limits. The same accounting applies here: `flagged_left_out` is never zero on a busy project, and no policy removes it.

The judge scores are proxies. A policy that reaches more flagged rows reaches more of what the judge believes, which is only as good as the judge. The human labels in Step 6 are the correction, and they only cover what the budget reached. Treat the agreement table as a calibration signal for the reviewed slice, not as a measurement of the judge on all traffic.

## Next steps

- [Annotation queues](https://langfuse.com/docs/evaluation/evaluation-methods/annotation-queues): assigning annotators, keyboard workflow, score configs.
- [LLM-as-a-Judge](https://langfuse.com/docs/evaluation/evaluation-methods/llm-as-a-judge): producing the signals this notebook consumes, with sampling and filters.
- [Scores API v3](https://api.reference.langfuse.com/#tag/scoresv3): the endpoint behind `fetch_scores`.
- [review-router](https://github.com/XinyangWuEthz/review-router): the capacity simulation behind the two benchmark-derived policies, including waiting-time effects that a static budget does not show.